## 1. Environment Setup

In [1]:
import sys
import os
from pathlib import Path
import json
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Add src to path
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

# Import orchestrator
from orchestrator import (
    create_initial_state,
    create_workflow,
    run_workflow,
    get_state_summary,
)

# Import models
from models import LoanApplication, LendingDecision

print("✅ Environment setup complete")
print(f"📁 Project root: {project_root}")
print(f"🐍 Python version: {sys.version.split()[0]}")

✅ Environment setup complete
📁 Project root: /Users/ducnguyenhuu/Documents/GitHub/DucNguyen_LearningSpace/under-writing-loan
🐍 Python version: 3.11.9


## 2. Start MCP Server

The MCP (Model Context Protocol) server provides credit bureau data. We'll start it in the background before running the workflows.

In [2]:
import subprocess
import time
import requests
from pathlib import Path

# Check if MCP server is already running
def is_mcp_running():
    try:
        response = requests.get("http://localhost:8000/health", timeout=2)
        return response.status_code == 200
    except:
        return False

# Start MCP server if not running
if is_mcp_running():
    print("✅ MCP server already running at http://localhost:8000")
else:
    print("🚀 Starting MCP server...")
    
    # Start server in background
    mcp_process = subprocess.Popen(
        ["python3", "-m", "uvicorn", "src.mcp.server:app", "--host", "0.0.0.0", "--port", "8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        cwd=str(project_root)
    )
    
    # Wait for server to start
    max_retries = 10
    for i in range(max_retries):
        time.sleep(1)
        if is_mcp_running():
            print(f"✅ MCP server started successfully at http://localhost:8000")
            break
        print(f"   Waiting for server to start... ({i+1}/{max_retries})")
    else:
        print("⚠️  MCP server may not have started. Workflow will use direct DB queries as fallback.")

# Verify credit profiles are available
try:
    response = requests.get("http://localhost:8000/credit/111-11-1111", timeout=5)
    if response.status_code == 200:
        print(f"✅ Credit data accessible - Test profile: {response.json()['name']} (Score: {response.json()['credit_score']})")
    else:
        print("⚠️  Credit endpoint returned non-200 status. Using fallback mode.")
except Exception as e:
    print(f"⚠️  Could not verify MCP server: {e}")
    print("   Workflow will use direct SQLite queries as fallback.")

🚀 Starting MCP server...
✅ MCP server started successfully at http://localhost:8000
⚠️  Could not verify MCP server: 'name'
   Workflow will use direct SQLite queries as fallback.
✅ MCP server started successfully at http://localhost:8000
⚠️  Could not verify MCP server: 'name'
   Workflow will use direct SQLite queries as fallback.


## 3. Helper Functions

In [3]:
def print_decision_summary(state: dict, scenario_name: str):
    """
    Display formatted decision summary with color-coded status.
    """
    print("\n" + "="*80)
    print(f"📊 {scenario_name.upper()}")
    print("="*80)
    
    # Extract key information
    application_id = state.get('application_id', 'Unknown')
    summary = get_state_summary(state)
    
    # Applicant info
    if state.get('loan_application'):
        app = state['loan_application']
        print(f"\n👤 APPLICANT: {app.get('first_name')} {app.get('last_name')}")
        print(f"   Application ID: {application_id}")
        print(f"   SSN: {app.get('ssn')}")
        print(f"   Requested Amount: ${app.get('requested_amount', 0):,.0f}")
    
    # Credit information
    if state.get('credit_report'):
        credit = state['credit_report']
        print(f"\n💳 CREDIT PROFILE:")
        print(f"   Credit Score: {credit.get('credit_score', 'N/A')}")
        print(f"   Payment History: {credit.get('payment_history', 'N/A').upper()}")
        print(f"   Credit Utilization: {credit.get('credit_utilization', 0):.1f}%")
        print(f"   Derogatory Marks: {credit.get('derogatory_marks', 0)}")
    
    # Risk metrics - FIXED: Use correct field names from RiskAssessment model
    if state.get('risk_assessment'):
        risk = state['risk_assessment']
        print(f"\n📊 RISK METRICS:")
        print(f"   DTI Ratio: {risk.get('debt_to_income_ratio', 0):.2f}%")
        print(f"   LTV Ratio: {risk.get('loan_to_value_ratio', 0):.2f}%")
        print(f"   Risk Level: {risk.get('risk_level', 'N/A').upper()}")
        print(f"   Risk Score: {risk.get('risk_score', 0):.2f}")
    
    # Compliance
    if state.get('compliance_report'):
        compliance = state['compliance_report']
        is_compliant = compliance.get('is_compliant', False)
        status_icon = '✅' if is_compliant else '⚠️'
        print(f"\n{status_icon} COMPLIANCE:")
        print(f"   Status: {'COMPLIANT' if is_compliant else 'NON-COMPLIANT'}")
        violations = compliance.get('violations', [])
        if violations:
            print(f"   Violations: {len(violations)}")
            for v in violations[:3]:  # Show first 3
                print(f"      - {v}")
    
    # Final decision
    if state.get('lending_decision'):
        decision = state['lending_decision']
        status = decision.get('decision_status', 'unknown')
        
        # Color-coded status
        if status == 'approved':
            icon = '✅'
            label = 'APPROVED'
        elif status == 'conditional_approval':
            icon = '⚠️'
            label = 'CONDITIONAL APPROVAL'
        else:
            icon = '❌'
            label = 'DENIED'
        
        print(f"\n{icon} FINAL DECISION: {label}")
        print(f"   Confidence: {decision.get('confidence_score', 0):.1%}")
        
        if decision.get('approved_amount'):
            print(f"   Approved Amount: ${decision.get('approved_amount', 0):,.0f}")
        
        if decision.get('interest_rate'):
            print(f"   Interest Rate: {decision.get('interest_rate', 0):.3f}%")
        
        if decision.get('monthly_payment'):
            print(f"   Monthly Payment: ${decision.get('monthly_payment', 0):,.2f}")
        
        # Reasons/conditions
        reasons = decision.get('reasons', [])
        if reasons:
            print(f"\n   {'Conditions' if status == 'conditional_approval' else 'Reasons'}:")
            for i, reason in enumerate(reasons[:5], 1):  # Show first 5
                print(f"      {i}. {reason}")
    
    # Execution metrics
    print(f"\n⏱️  EXECUTION METRICS:")
    print(f"   Total Duration: {summary.get('total_duration', 0):.2f}s")
    print(f"   Total Tokens: {summary.get('total_tokens', 0):,}")
    print(f"   Total Cost: ${summary.get('total_cost', 0):.4f}")
    print(f"   Errors: {summary.get('error_count', 0)}")
    
    print("\n" + "="*80 + "\n")


def create_comparison_chart(results: list):
    """
    Create interactive comparison chart for all three scenarios.
    
    Args:
        results: List of dicts with scenario results
    """
    df = pd.DataFrame(results)
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Credit Scores',
            'Risk Ratios (DTI, LTV)',
            'Risk Scores',
            'Final Decisions'
        ),
        specs=[
            [{'type': 'bar'}, {'type': 'bar'}],
            [{'type': 'bar'}, {'type': 'bar'}]
        ]
    )
    
    # 1. Credit Scores
    fig.add_trace(
        go.Bar(
            x=df['scenario'],
            y=df['credit_score'],
            name='Credit Score',
            marker_color=['green', 'orange', 'red'],
            text=df['credit_score'],
            textposition='outside'
        ),
        row=1, col=1
    )
    
    # 2. Risk Ratios
    fig.add_trace(
        go.Bar(x=df['scenario'], y=df['dti'], name='DTI', marker_color='lightblue'),
        row=1, col=2
    )
    fig.add_trace(
        go.Bar(x=df['scenario'], y=df['ltv'], name='LTV', marker_color='lightgreen'),
        row=1, col=2
    )
    
    # 3. Risk Scores
    fig.add_trace(
        go.Bar(
            x=df['scenario'],
            y=df['risk_score'],
            name='Risk Score',
            marker_color=['lightgreen', 'lightyellow', 'lightcoral'],
            text=df['risk_score'].round(2),
            textposition='outside'
        ),
        row=2, col=1
    )
    
    # 4. Decision Status (encoded as numeric)
    decision_map = {'approved': 3, 'conditional_approval': 2, 'denied': 1}
    df['decision_numeric'] = df['decision'].map(decision_map)
    
    fig.add_trace(
        go.Bar(
            x=df['scenario'],
            y=df['decision_numeric'],
            name='Decision',
            marker_color=['green', 'orange', 'red'],
            text=df['decision'].str.replace('_', ' ').str.upper(),
            textposition='outside'
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        height=800,
        showlegend=True,
        title_text="Underwriting Scenarios Comparison",
        title_font_size=20
    )
    
    # Y-axis labels
    fig.update_yaxes(title_text="Score", row=1, col=1)
    fig.update_yaxes(title_text="Percentage (%)", row=1, col=2)
    fig.update_yaxes(title_text="Risk Score", row=2, col=1)
    fig.update_yaxes(title_text="Status", row=2, col=2)
    
    fig.show()

print("✅ Helper functions loaded")


✅ Helper functions loaded


---

# SCENARIO 1: AUTO-APPROVAL ✅

## Profile: Excellent Borrower
- **Credit Score**: 780 (Excellent)
- **DTI**: ~20% (Very low debt)
- **LTV**: 60% (Strong down payment)
- **Income**: $180,000/year
- **Property Value**: $500,000
- **Loan Amount**: $300,000

**Expected Outcome**: ✅ **APPROVED** - All criteria strongly met, minimal risk

In [4]:
# Scenario 1: Auto-Approval - Excellent Profile
scenario1_application = {
    "application_id": "APP-2025-101",
    "first_name": "Alice",
    "last_name": "Excellence",
    "ssn": "111-11-1111",  # Maps to 780 credit score in mock DB
    "email": "alice.excellence@example.com",
    "phone": "+1-555-0101",
    "date_of_birth": "1985-03-15",
    "annual_income": 180000,  # High income
    "employment_status": "employed",
    "employer_name": "Tech Corp",
    "years_at_current_job": 8,
    "monthly_debt_payments": 3000,  # Low debt (DTI ~20%)
    "requested_amount": 300000,
    "loan_purpose": "home_purchase",
    "property_address": "456 Elite Ave, San Francisco, CA 94102",
    "property_value": 500000,  # LTV = 60%
    "down_payment": 200000,  # 40% down
    "document_paths": [
        str(project_root / "data" / "applications" / "paystub_scenario1_approval.pdf")
    ]
}

print("📋 SCENARIO 1: AUTO-APPROVAL")
print(f"   Applicant: {scenario1_application['first_name']} {scenario1_application['last_name']}")
print(f"   Credit Score: 780 (Excellent)")
print(f"   Annual Income: ${scenario1_application['annual_income']:,}")
print(f"   Loan Amount: ${scenario1_application['requested_amount']:,}")
print(f"   Property Value: ${scenario1_application['property_value']:,}")
print(f"   Expected LTV: {(scenario1_application['requested_amount'] / scenario1_application['property_value'] * 100):.1f}%")
print(f"   Expected DTI: ~{(scenario1_application['monthly_debt_payments'] * 12 / scenario1_application['annual_income'] * 100):.1f}%")
print(f"\n🚀 Running workflow...")

📋 SCENARIO 1: AUTO-APPROVAL
   Applicant: Alice Excellence
   Credit Score: 780 (Excellent)
   Annual Income: $180,000
   Loan Amount: $300,000
   Property Value: $500,000
   Expected LTV: 60.0%
   Expected DTI: ~20.0%

🚀 Running workflow...


In [5]:
# Execute workflow for Scenario 1
state1 = run_workflow(scenario1_application)
print_decision_summary(state1, "Scenario 1: Auto-Approval")

INFO:orchestrator:📄 Document Agent starting for application APP-2025-101
INFO:agents.document_agent:DocumentIntelligenceExtractor initialized with endpoint: https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/
INFO:orchestrator:  Analyzing document: /Users/ducnguyenhuu/Documents/GitHub/DucNguyen_LearningSpace/under-writing-loan/data/applications/paystub_scenario1_approval.pdf
INFO:agents.document_agent:Analyzing document: paystub_scenario1_approval.pdf (type: pay_stub)
INFO:agents.document_agent:Calling Document Intelligence with model: prebuilt-read
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-read:analyze?stringIndexType=unicodeCodePoint&api-version=2023-07-31'
Request method: 'POST'
Request headers:
    'Content-Length': '2106'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/json'
    'x-ms-client-request-id


📊 SCENARIO 1: AUTO-APPROVAL

👤 APPLICANT: Alice Excellence
   Application ID: APP-2025-101
   SSN: 111-11-1111
   Requested Amount: $300,000

💳 CREDIT PROFILE:
   Credit Score: 780
   Payment History: EXCELLENT
   Credit Utilization: 15.0%
   Derogatory Marks: 0

📊 RISK METRICS:
   DTI Ratio: 20.00%
   LTV Ratio: 60.00%
   Risk Level: LOW
   Risk Score: 34.62

✅ COMPLIANCE:
   Status: COMPLIANT

❌ FINAL DECISION: DENIED
   Confidence: 0.0%
   Approved Amount: $300,000
   Interest Rate: 6.000%
   Monthly Payment: $1,798.65

⏱️  EXECUTION METRICS:
   Total Duration: 12.96s
   Total Tokens: 6,500
   Total Cost: $0.0620
   Errors: 0




---

# SCENARIO 2: CONDITIONAL APPROVAL ⚠️

## Profile: Marginal Borrower (Needs Review)
- **Credit Score**: 720 (Good, but not excellent)
- **DTI**: ~38% (At threshold)
- **LTV**: 85% (High, minimal down payment)
- **Income**: $95,000/year
- **Property Value**: $400,000
- **Loan Amount**: $340,000

**Expected Outcome**: ⚠️ **CONDITIONAL APPROVAL** - Marginal metrics, may require:
- Additional documentation
- Higher interest rate
- PMI (Private Mortgage Insurance)
- Manual underwriter review

In [6]:
# Scenario 2: Conditional Approval - Marginal Profile
scenario2_application = {
    "application_id": "APP-2025-202",
    "first_name": "Bob",
    "last_name": "Marginal",
    "ssn": "222-22-2222",  # Maps to 720 credit score in mock DB
    "email": "bob.marginal@example.com",
    "phone": "+1-555-0202",
    "date_of_birth": "1988-07-22",
    "annual_income": 95000,  # Moderate income
    "employment_status": "employed",
    "employer_name": "Small Business Inc",
    "years_at_current_job": 3,
    "monthly_debt_payments": 3000,  # High debt (DTI ~38%)
    "requested_amount": 340000,
    "loan_purpose": "home_purchase",
    "property_address": "789 Borderline Blvd, Oakland, CA 94601",
    "property_value": 400000,  # LTV = 85%
    "down_payment": 60000,  # 15% down (minimal)
    "document_paths": [
        str(project_root / "data" / "applications" / "paystub_scenario2_conditional.pdf")
    ]
}

print("📋 SCENARIO 2: CONDITIONAL APPROVAL")
print(f"   Applicant: {scenario2_application['first_name']} {scenario2_application['last_name']}")
print(f"   Credit Score: 720 (Good)")
print(f"   Annual Income: ${scenario2_application['annual_income']:,}")
print(f"   Loan Amount: ${scenario2_application['requested_amount']:,}")
print(f"   Property Value: ${scenario2_application['property_value']:,}")
print(f"   Expected LTV: {(scenario2_application['requested_amount'] / scenario2_application['property_value'] * 100):.1f}%")
print(f"   Expected DTI: ~{(scenario2_application['monthly_debt_payments'] * 12 / scenario2_application['annual_income'] * 100):.1f}%")
print(f"\n🚀 Running workflow...")

📋 SCENARIO 2: CONDITIONAL APPROVAL
   Applicant: Bob Marginal
   Credit Score: 720 (Good)
   Annual Income: $95,000
   Loan Amount: $340,000
   Property Value: $400,000
   Expected LTV: 85.0%
   Expected DTI: ~37.9%

🚀 Running workflow...


In [7]:
# Execute workflow for Scenario 2
state2 = run_workflow(scenario2_application)
print_decision_summary(state2, "Scenario 2: Conditional Approval")

INFO:orchestrator:Creating initial state for application APP-2025-202
INFO:orchestrator:🚀 Starting workflow for application APP-2025-202
INFO:orchestrator:🚀 Starting workflow for application APP-2025-202
INFO:orchestrator:✅ LangGraph workflow created successfully
INFO:orchestrator:   Nodes: document → risk → compliance → decision
INFO:orchestrator:   Error handling: Conditional routing after each node
INFO:orchestrator:📄 Document Agent starting for application APP-2025-202
INFO:agents.document_agent:DocumentIntelligenceExtractor initialized with endpoint: https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/
INFO:orchestrator:  Analyzing document: /Users/ducnguyenhuu/Documents/GitHub/DucNguyen_LearningSpace/under-writing-loan/data/applications/paystub_scenario2_conditional.pdf
INFO:agents.document_agent:Analyzing document: paystub_scenario2_conditional.pdf (type: pay_stub)
INFO:agents.document_agent:Calling Document Intelligence with model: prebuilt-read
INFO:azure.co


📊 SCENARIO 2: CONDITIONAL APPROVAL

👤 APPLICANT: Bob Marginal
   Application ID: APP-2025-202
   SSN: 222-22-2222
   Requested Amount: $340,000

💳 CREDIT PROFILE:
   Credit Score: 720
   Payment History: GOOD
   Credit Utilization: 28.5%
   Derogatory Marks: 0

📊 RISK METRICS:
   DTI Ratio: 37.89%
   LTV Ratio: 85.00%
   Risk Level: MEDIUM
   Risk Score: 60.69

⚠️ COMPLIANCE:
   Status: NON-COMPLIANT
   Violations: 3
      - {'policy_name': 'Underwriting Standards', 'policy_section': '3.1', 'severity': 'critical', 'description': 'The Debt-to-Income (DTI) ratio of 37.89% exceeds the maximum allowable DTI of 36%, indicating a higher proportion of income allocated to debt obligations.', 'recommendation': 'Consider reducing existing debt or increasing income to bring the DTI below the threshold.'}
      - {'policy_name': 'Underwriting Standards', 'policy_section': '4.1', 'severity': 'warning', 'description': 'The Loan-to-Value (LTV) ratio of 85.00% exceeds the no PMI threshold of 80%, sug

---

# SCENARIO 3: REJECTION ❌

## Profile: High-Risk Borrower
- **Credit Score**: 590 (Poor)
- **DTI**: ~55% (Far exceeds 43% threshold)
- **LTV**: 95% (Minimal down payment)
- **Income**: $55,000/year
- **Property Value**: $300,000
- **Loan Amount**: $285,000
- **Credit Issues**: Multiple derogatory marks, high utilization

**Expected Outcome**: ❌ **DENIED** - Multiple risk factors:
- DTI >43% AND Credit Score <640 (auto-reject rule)
- Poor payment history
- High credit utilization (85%)
- Multiple derogatory marks

In [8]:
# Scenario 3: Rejection - High-Risk Profile
scenario3_application = {
    "application_id": "APP-2025-303",
    "first_name": "Charlie",
    "last_name": "Risky",
    "ssn": "444-44-4444",  # Maps to 590 credit score in mock DB
    "email": "charlie.risky@example.com",
    "phone": "+1-555-0303",
    "date_of_birth": "1992-11-30",
    "annual_income": 55000,  # Low income
    "employment_status": "employed",
    "employer_name": "Gig Economy Co",
    "years_at_current_job": 1,
    "monthly_debt_payments": 2500,  # Very high debt (DTI ~55%)
    "requested_amount": 285000,
    "loan_purpose": "home_purchase",
    "property_address": "321 Struggle St, Fresno, CA 93650",
    "property_value": 300000,  # LTV = 95%
    "down_payment": 15000,  # 5% down (very minimal)
    "document_paths": [
        str(project_root / "data" / "applications" / "paystub_scenario3_rejection.pdf")
    ]
}

print("📋 SCENARIO 3: REJECTION")
print(f"   Applicant: {scenario3_application['first_name']} {scenario3_application['last_name']}")
print(f"   Credit Score: 590 (Poor)")
print(f"   Annual Income: ${scenario3_application['annual_income']:,}")
print(f"   Loan Amount: ${scenario3_application['requested_amount']:,}")
print(f"   Property Value: ${scenario3_application['property_value']:,}")
print(f"   Expected LTV: {(scenario3_application['requested_amount'] / scenario3_application['property_value'] * 100):.1f}%")
print(f"   Expected DTI: ~{(scenario3_application['monthly_debt_payments'] * 12 / scenario3_application['annual_income'] * 100):.1f}%")
print(f"\n🚀 Running workflow...")

📋 SCENARIO 3: REJECTION
   Applicant: Charlie Risky
   Credit Score: 590 (Poor)
   Annual Income: $55,000
   Loan Amount: $285,000
   Property Value: $300,000
   Expected LTV: 95.0%
   Expected DTI: ~54.5%

🚀 Running workflow...


In [9]:
# Execute workflow for Scenario 3
state3 = run_workflow(scenario3_application)
print_decision_summary(state3, "Scenario 3: Rejection")

INFO:orchestrator:Creating initial state for application APP-2025-303
INFO:orchestrator:🚀 Starting workflow for application APP-2025-303
INFO:orchestrator:🚀 Starting workflow for application APP-2025-303
INFO:orchestrator:✅ LangGraph workflow created successfully
INFO:orchestrator:   Nodes: document → risk → compliance → decision
INFO:orchestrator:   Error handling: Conditional routing after each node
INFO:orchestrator:📄 Document Agent starting for application APP-2025-303
INFO:agents.document_agent:DocumentIntelligenceExtractor initialized with endpoint: https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/
INFO:orchestrator:  Analyzing document: /Users/ducnguyenhuu/Documents/GitHub/DucNguyen_LearningSpace/under-writing-loan/data/applications/paystub_scenario3_rejection.pdf
INFO:agents.document_agent:Analyzing document: paystub_scenario3_rejection.pdf (type: pay_stub)
INFO:agents.document_agent:Calling Document Intelligence with model: prebuilt-read
INFO:azure.core.p


📊 SCENARIO 3: REJECTION

👤 APPLICANT: Charlie Risky
   Application ID: APP-2025-303
   SSN: 444-44-4444
   Requested Amount: $285,000

💳 CREDIT PROFILE:
   Credit Score: 590
   Payment History: POOR
   Credit Utilization: 85.0%
   Derogatory Marks: 3

📊 RISK METRICS:
   DTI Ratio: 54.55%
   LTV Ratio: 95.00%
   Risk Level: HIGH
   Risk Score: 78.91

⚠️ COMPLIANCE:
   Status: NON-COMPLIANT
   Violations: 3
      - {'policy_name': 'Underwriting Standards', 'policy_section': 'Section 2.1', 'severity': 'critical', 'description': "The applicant's credit score of 590 is below the minimum acceptable credit score of 620, which requires automatic decline unless extraordinary compensating factors are present.", 'recommendation': 'Decline the application due to insufficient credit score unless strong compensating factors can be documented.'}
      - {'policy_name': 'Underwriting Standards', 'policy_section': 'Section 3.1', 'severity': 'critical', 'description': 'The Debt-to-Income (DTI) ratio of

---

# Comparative Analysis

Compare all three scenarios side-by-side to understand how different risk factors impact the final decision.

In [10]:
# Extract comparison data - FIXED: Use correct field names
results = []

for scenario_name, state in [("Scenario 1: Auto-Approval", state1), 
                             ("Scenario 2: Conditional", state2), 
                             ("Scenario 3: Rejection", state3)]:
    result = {
        'scenario': scenario_name,
        'credit_score': state.get('credit_report', {}).get('credit_score', 0),
        'dti': state.get('risk_assessment', {}).get('debt_to_income_ratio', 0),
        'ltv': state.get('risk_assessment', {}).get('loan_to_value_ratio', 0),
        'risk_score': state.get('risk_assessment', {}).get('risk_score', 0),
        'decision': state.get('lending_decision', {}).get('decision_status', 'unknown'),
        'confidence': state.get('lending_decision', {}).get('confidence_score', 0),
        'interest_rate': state.get('lending_decision', {}).get('interest_rate', 0),
    }
    results.append(result)

# Create comparison DataFrame
comparison_df = pd.DataFrame(results)
print("\n" + "="*100)
print("SCENARIO COMPARISON TABLE")
print("="*100)
print(comparison_df.to_string(index=False))
print("="*100)

# Create interactive visualization
create_comparison_chart(results)



SCENARIO COMPARISON TABLE
                 scenario  credit_score   dti   ltv  risk_score decision  confidence interest_rate
Scenario 1: Auto-Approval           780 20.00 60.00       34.62  unknown           0           6.0
  Scenario 2: Conditional           720 37.89 85.00       60.69  unknown           0          6.65
    Scenario 3: Rejection           590 54.55 95.00       78.91  unknown           0          None


## Key Insights

### Decision Rules Applied:

1. **Auto-Reject Rule**: DTI >43% **AND** Credit Score <640
   - Scenario 3 triggered this rule (DTI ~55%, Score 590)

2. **High-Risk Flags**:
   - DTI >38% → Scenarios 2 & 3
   - LTV >80% → Scenarios 2 & 3
   - Credit Score <680 → Scenario 3

3. **Auto-Approve Criteria** (All must be met):
   - Credit Score >740 → ✅ Scenario 1 only
   - DTI <35% → ✅ Scenario 1 only
   - LTV <75% → ✅ Scenario 1 only

### Practical Implications:

- **Scenario 1**: Strong applicant, instant approval with best rate
- **Scenario 2**: Marginal case, requires manual review, may need PMI, higher rate
- **Scenario 3**: High risk, automatic denial, applicant should improve credit/reduce debt before reapplying

---

## Execution Cost Summary

In [11]:
# Calculate total costs across all scenarios
total_duration = sum([get_state_summary(s).get('total_duration', 0) for s in [state1, state2, state3]])
total_tokens = sum([get_state_summary(s).get('total_tokens', 0) for s in [state1, state2, state3]])
total_cost = sum([get_state_summary(s).get('total_cost', 0) for s in [state1, state2, state3]])

print("\n" + "="*80)
print("TOTAL EXECUTION SUMMARY (3 Applications)")
print("="*80)
print(f"Total Duration: {total_duration:.2f}s ({total_duration/60:.2f} minutes)")
print(f"Total Tokens: {total_tokens:,}")
print(f"Total Cost: ${total_cost:.4f}")
print(f"\nCost Per Application: ${total_cost/3:.4f}")
print(f"Time Per Application: {total_duration/3:.2f}s")
print("="*80)


TOTAL EXECUTION SUMMARY (3 Applications)
Total Duration: 63.71s (1.06 minutes)
Total Tokens: 19,500
Total Cost: $0.1860

Cost Per Application: $0.0620
Time Per Application: 21.24s
